# Сборка показа: 64 трека, без повторного восприятия
Основа: 8afe7d0d4c465dc51ac795c4e418bc598bef1a93.
Обычная среда CPU подходит; GPU и Colab Pro не обязательны. Детектор, ReID, склейка и перебор порогов не запускаются.
На Drive/MyDrive/mct загрузите mct_demo_build_64.zip. Пять исходных camera_new_*.mkv должны уже лежать в этой папке.
Выполните все ячейки. Итоговый архив будет сохранён на Drive. Длительность зависит от выделенного CPU.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, sys, subprocess, zipfile, json, hashlib, shutil
from pathlib import Path
DRIVE = Path('/content/drive/MyDrive/mct')
ROOT = Path('/content/mct_demo_64')
ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(DRIVE/'mct_demo_build_64.zip') as archive:
    for member in archive.namelist():
        assert (ROOT/member).resolve().is_relative_to(ROOT.resolve()), member
    archive.extractall(ROOT)
bundle = json.loads((ROOT/'demo_bundle.json').read_text())
assert bundle['total_tracks'] == 64
for name, expected in bundle['files'].items():
    assert hashlib.sha256((ROOT/name).read_bytes()).hexdigest() == expected, name
print('Комплект проверен:', bundle['base_commit'], bundle['total_tracks'], 'трека')
subprocess.run([sys.executable,'-m','pip','install','-q','opencv-python==4.11.0.86','numpy','pandas','pyarrow','pycocotools','pyyaml','pillow','matplotlib','scipy','plotly','imageio-ffmpeg'],check=True)
os.chdir(ROOT)
os.environ['PYTHONPATH'] = str(ROOT/'src') + os.pathsep + str(ROOT)
os.environ['PYTHONUTF8'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
VIDEOS = Path('/content/mct_demo_videos')
VIDEOS.mkdir(exist_ok=True)
for camera in range(1,6):
    name=f'camera_new_{camera}.mkv'
    source=DRIVE/name
    assert source.is_file(), f'Нет исходной записи: {source}'
    target=VIDEOS/name
    if not target.exists() or target.stat().st_size != source.stat().st_size:
        shutil.copy2(source,target)
print('Исходные видео скопированы на локальный диск Colab')


## Сборка и проверки
Этапы: мозаика → короткий ролик → HTML-отчёт → 10 проверок. Завершённые этапы проверяются по хешам и пропускаются при повторном запуске. При перезапуске всей среды локальные файлы могут исчезнуть: тогда сборка повторится.


In [ ]:
OUT = ROOT/'render/full'
subprocess.run([sys.executable,'scripts/build_demo_assets.py','--videos',str(VIDEOS),'--out',str(OUT)],check=True)
print((OUT/'build_check.log').read_text())


## Сохранение на Drive
Архив содержит готовый показ, перечень треков и журнал сборки. Дождитесь сообщения «Готово» перед отключением среды.


In [ ]:
from IPython.display import display, Video
state=json.loads((OUT/'demo_build_state.json').read_text())
assert 'check' in state['stages'], 'Нет успешной финальной проверки'
archive_path=DRIVE/'demo_ready_64.zip'
with zipfile.ZipFile(archive_path,'w',zipfile.ZIP_STORED) as z:
    for folder in [ROOT/'render/full',ROOT/'results']:
        for file in sorted(folder.rglob('*')):
            if file.is_file():z.write(file,file.relative_to(ROOT))
    z.write(ROOT/'demo_bundle.json','demo_bundle.json')
print('Готово:', archive_path, round(archive_path.stat().st_size/1e6,1), 'МБ')
display(Video(str(OUT/'showcase.mp4'),embed=True,width=960))
